In [1]:
from markov_models import MarkovModel 
from helpers import compute_info_rate, update_values_in_csv, check_data_availability
import numpy as np
import pickle

languages = ["FRA"] # ["FRA", 'JPN', 'CMN', 'VIE', 'YUE', 'ENG', 'DEU']

for language in languages:

    for processing_type in ['phones','sylls']: 
        print(f"\nLanguage: {language}")
        print(f"Processing type: {processing_type.upper()}")
        print(f"=======================================================================================")

        input_path = check_data_availability(language, processing_type)
        if input_path: 
            with open(input_path, "rb") as f:
                data = pickle.load(f)
                print(data[:10])
            
        else: continue
        
        for text_type in ['words', 'sentences']:
            print(f"\n📊 Computing ID and IR across {text_type.upper()}")

            n_values = [1, 2, 3, 4]  # For bigram, trigram, and quadgram models
            markov_models = {}

            for n in n_values:

                print(f"\n🧮 Training a Markov Model with n = {n}:")

                # Create and build the Markov model
                model = MarkovModel(n)

                # Build the markov model
                model.build(data, text_type)

                # Compute the conditional entropy (information density)
                info_density = model.compute_conditional_entropy()
                print(f"Information Density: {info_density:.4f}")

                # Compute the information rate (bits per second)
                info_rates = compute_info_rate(info_density, processing_type, language)
                print(f"Information Rate: {np.mean(info_rates):.4f}")
                
                # Update the CSV file with the computed values
                update_values_in_csv(language, info_density, n, 'ID', text_type)
                update_values_in_csv(language, info_rates, n, 'IR', text_type)

                # Store model for later use 
                markov_models[n] = model

                # Display exactly 3 examples
                """example_count = 0
                print("\nExample probabilities (p(x, y)):")

                for (prefix, suffix), p_xy in model.cond_probs.items():
                    print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
                    example_count += 1
                    if example_count == 3:
                        break"""
                
                # Save the model to a file
                model.save_model(language, processing_type, text_type)

    # For plotting, see plotting.ipynb


Language: FRA
Processing type: PHONES
Checking data availability...
❌ No prepared phones data found for FRA at produced_data\FRA\phones\phonized_FRA.pkl.
📄 However, raw sentence-level corpus found at Z:/data/FRA/fr.tok. 
👉 Please run `parse_to_phones_and_sylls('FRA')` to generate the required data.
❓ Do you want to run it now? [y/n]:
> y
📥 Loading corpus data ...
✅ Loaded 570599 sentences from Z:/data/FRA/fr.tok
Language: FRA
Loading CharsiuG2P model: charsiu/g2p_multilingual_byT5_tiny_16_layers_100...
CharsiuG2P model running on CPU.
Total words processed: 17238
Onsets for FRA saved to 'produced_data/FRA\FRA_ipa_sentences.pkl'


AttributeError: 'list' object has no attribute 'lower'